In [6]:
# Gregory-Newton Forward Difference Table with Explicit Calculations

from fractions import Fraction
import math

# ---------------------------------------------------
# 1) Construir tabela numérica de diferenças
# ---------------------------------------------------
def build_diff_table_values(y_vals):
    """
    Retorna a tabela de diferenças como lista de colunas:
    diffs[0] = f(x_k)
    diffs[1] = Δ¹ f(x_k)
    diffs[2] = Δ² f(x_k)
    ...
    """
    n = len(y_vals)
    diffs = [y_vals[:]]  # cópia
    for k in range(1, n):
        prev = diffs[k-1]
        col = []
        for i in range(len(prev) - 1):
            col.append(prev[i+1] - prev[i])
        diffs.append(col)
    return diffs

# ---------------------------------------------------
# 2) Imprimir tabela "estilo livro", com as contas
# ---------------------------------------------------
def print_diff_table_explicit(x_vals, y_vals):
    n = len(x_vals)
    diffs = build_diff_table_values(y_vals)
    num_orders = len(diffs) - 1  # até Δ^{n-1}

    # table[r][c] = texto que vai naquela célula
    cols = 1 + len(diffs)  # x_k + Δ^0 + ... + Δ^{n-1}
    table = [[None] * cols for _ in range(n)]

    # primeira coluna: x_k
    for i in range(n):
        table[i][0] = str(x_vals[i])

    # segunda coluna: Δ^0 f(x_k) = f(x_k)
    for i in range(n):
        table[i][1] = str(y_vals[i])

    # demais colunas: diferenças com a conta explícita
    for k in range(1, len(diffs)):  # k = 1 .. n-1
        prev = diffs[k-1]
        col = diffs[k]
        for i in range(len(col)):
            a = prev[i+1]
            b = prev[i]
            # só coloca parênteses se for negativo, para ficar bonito
            if b < 0:
                expr = f"{a} - ({b}) = {col[i]}"
            else:
                expr = f"{a} - {b} = {col[i]}"
            table[i][k+1] = expr

    # nomes das colunas
    col_names = ["x_k", "Δ⁰ f(x_k)"]
    for k in range(1, len(diffs)):
        col_names.append(f"Δ^{k} f(x_k)")

    # calcular larguras
    col_widths = []
    for c in range(cols):
        max_len = len(col_names[c])
        for r in range(n):
            if table[r][c] is not None:
                max_len = max(max_len, len(table[r][c]))
        col_widths.append(max_len + 2)

    # cabeçalho
    header = ""
    for c in range(cols):
        header += col_names[c].ljust(col_widths[c])
    print(header)
    print("-" * sum(col_widths))

    # linhas
    for r in range(n):
        row = ""
        for c in range(cols):
            cell = table[r][c] if table[r][c] is not None else ""
            row += cell.ljust(col_widths[c])
        print(row)


# ---------------------------------------------------
# 3) Ferramentas para polinômios (lista de coeficientes)
# ---------------------------------------------------
def poly_mul_x_minus_a(p, a):
    """
    Multiplica um polinômio p(x) por (x - a).
    p = [c0, c1, c2, ...]  (c0 + c1 x + c2 x^2 + ...)
    """
    res = [Fraction(0)] * (len(p) + 1)
    for i, coef in enumerate(p):
        res[i+1] += coef        # vezes x
        res[i]   += -a * coef   # vezes (-a)
    return res

def poly_add(p, q):
    m = max(len(p), len(q))
    res = [Fraction(0)] * m
    for i in range(m):
        if i < len(p):
            res[i] += p[i]
        if i < len(q):
            res[i] += q[i]
    return res

def poly_scalar_mul(p, c):
    return [coef * c for coef in p]

def poly_to_string(p):
    """
    Converte [c0, c1, ..., cn] em string tipo "5*x^3 - 4*x + 30".
    """
    terms = []
    for power in range(len(p) - 1, -1, -1):
        coef = p[power]
        if coef == 0:
            continue

        sign = '+' if coef > 0 else '-'
        abs_coef = abs(coef)

        # coeficiente em texto
        if abs_coef.denominator == 1:
            c_str = str(abs_coef.numerator)
        else:
            c_str = f"{abs_coef.numerator}/{abs_coef.denominator}"

        if power == 0:
            term = c_str
        elif power == 1:
            term = "x" if abs_coef == 1 else f"{c_str}*x"
        else:
            term = f"x^{power}" if abs_coef == 1 else f"{c_str}*x^{power}"

        terms.append((sign, term))

    if not terms:
        return "0"

    first_sign, first_term = terms[0]
    expr = ""
    if first_sign == '-':
        expr += '-'
    expr += first_term

    for sign, term in terms[1:]:
        expr += f" {sign} {term}"

    return expr

# ---------------------------------------------------
# 4) Gregory–Newton passo a passo (forma progressiva)
# ---------------------------------------------------
def gregory_newton_step_by_step(x_vals, y_vals):
    x0 = x_vals[0]
    h = x_vals[1] - x0

    # tabela de diferenças (numérica) e depois em Fraction
    diffs_vals = build_diff_table_values(y_vals)
    diffs = [list(map(Fraction, col)) for col in diffs_vals]

    print("\n=== Tabela de diferenças (apenas valores) ===")
    for k, col in enumerate(diffs):
        print(f"Δ^{k}:", [float(c) for c in col])

    # P_0(x) = f(x0)
    P = [Fraction(diffs[0][0])]
    print("\n=== Passo 0 ===")
    print(f"P_0(x) = f(x0) = {poly_to_string(P)}")

    # base B_0(x) = 1
    B = [Fraction(1)]

    # termos de ordem 1..n-1
    for k in range(1, len(x_vals)):
        print(f"\n=== Passo {k}: termo de ordem {k} ===")
        delta_k0 = diffs[k][0]
        print(f"Δ^{k} f(x0) = {delta_k0}")

        # atualiza base B_k(x) = B_{k-1}(x) * (x - x_{k-1})
        B = poly_mul_x_minus_a(B, Fraction(x_vals[k-1]))
        print(f"B_{k}(x) = ∏(x - x_i), i=0..{k-1} = {poly_to_string(B)}")

        # coeficiente c_k
        c = delta_k0 / (Fraction(math.factorial(k)) * (Fraction(h) ** k))
        print(f"c_{k} = Δ^{k} f(x0) / ({k}! * h^{k})")
        print(f"    = {delta_k0} / ({math.factorial(k)} * {h}^{k}) = {c}")

        # termo T_k(x) = c_k * B_k(x)
        term = poly_scalar_mul(B, c)
        print(f"T_{k}(x) = c_{k} * B_{k}(x) = {poly_to_string(term)}")

        # P_k(x) = P_{k-1}(x) + T_k(x)
        P = poly_add(P, term)
        print(f"P_{k}(x) = P_{k-1}(x) + T_{k}(x) = {poly_to_string(P)}")

    print("\n=== Polinômio final ===")
    print("P(x) =", poly_to_string(P))
    return P

if __name__ == "__main__":
    xs = [-9, -4, 1, 6, 11]     
    ys = [-808, -78, 2, 182, 1212] 

    print("TABELA DE DIFERENÇAS (ESTILO LIVRO):\n")
    print_diff_table_explicit(xs, ys)

    print("\n\nCONSTRUÇÃO DO POLINÔMIO DE GREGORY–NEWTON:\n")
    gregory_newton_step_by_step(xs, ys)


TABELA DE DIFERENÇAS (ESTILO LIVRO):

x_k  Δ⁰ f(x_k)  Δ^1 f(x_k)          Δ^2 f(x_k)        Δ^3 f(x_k)          Δ^4 f(x_k)     
-----------------------------------------------------------------------------------------
-9   -808       -78 - (-808) = 730  80 - 730 = -650   100 - (-650) = 750  750 - 750 = 0  
-4   -78        2 - (-78) = 80      180 - 80 = 100    850 - 100 = 750                    
1    2          182 - 2 = 180       1030 - 180 = 850                                     
6    182        1212 - 182 = 1030                                                        
11   1212                                                                                


CONSTRUÇÃO DO POLINÔMIO DE GREGORY–NEWTON:


=== Tabela de diferenças (apenas valores) ===
Δ^0: [-808.0, -78.0, 2.0, 182.0, 1212.0]
Δ^1: [730.0, 80.0, 180.0, 1030.0]
Δ^2: [-650.0, 100.0, 850.0]
Δ^3: [750.0, 750.0]
Δ^4: [0.0]

=== Passo 0 ===
P_0(x) = f(x0) = -808

=== Passo 1: termo de ordem 1 ===
Δ^1 f(x0) = 730
B_1(x) = ∏(x - 

In [7]:
# -*- coding: utf-8 -*-
from sympy import symbols, Rational, factorial, expand, latex, fraction
from IPython.display import display, Math


def _tabela_diferencas(ys):
    """Gera tabela de diferenças progressivas."""
    tabela = [ys]
    while len(tabela[-1]) > 1:
        ant = tabela[-1]
        nova = [ant[i+1] - ant[i] for i in range(len(ant) - 1)]
        tabela.append(nova)
    return tabela


def gregory_newton_progressivo_latex(xs, ys, var_name="x"):
    """
    Interpolação de Gregory–Newton progressiva com saída em LaTeX.

    xs: lista de x_i (igualmente espaçados)
    ys: lista de f(x_i)
    var_name: nome da variável (ex: "x")
    """
    # 1) Simboliza e converte para frações exatas
    x = symbols(var_name)
    xs = [Rational(v) for v in xs]
    ys = [Rational(v) for v in ys]

    n = len(xs)

    # 2) Verifica h constante
    if n < 2:
        raise ValueError("Preciso de pelo menos 2 pontos.")
    h = xs[1] - xs[0]
    for i in range(1, n-1):
        if xs[i+1] - xs[i] != h:
            raise ValueError("Os x_i devem ser igualmente espaçados (mesmo h).")

    # 3) Tabela de diferenças
    tabela = _tabela_diferencas(ys)

    print("Pontos de interpolação:")
    for i, (xi, yi) in enumerate(zip(xs, ys)):
        print(f"  x_{i} = {xi},  f(x_{i}) = {yi}")
    print(f"\nPasso h = {h}\n")

    print("Tabela de diferenças progressivas (somente primeira coluna):")
    for k, linha in enumerate(tabela):
        print(f"  Δ^{k} y_0 = {linha[0]}")
    print()

    # 4) Define s
    s = (x - xs[0]) / h
    display(Math(r"s = \dfrac{%s - %s}{%s}" % (latex(x), latex(xs[0]), latex(h))))

    # 5) Fórmula geral
    display(Math(
        r"P_{%d}(%s) = y_0 + \frac{\Delta y_0}{1!}s"
        r" + \frac{\Delta^2 y_0}{2!}s(s-1) + \cdots"
        % (n-1, var_name)
    ))

    P = 0  # polinômio final

    # 6) Constrói cada termo T_k(x)
    for k in range(n):
        Δk = tabela[k][0]

        # produto em s: (s)(s-1)...(s-k+1)
        prod_s = 1
        prod_s_tex = ""
        for j in range(k):
            prod_s *= (s - j)
            prod_s_tex += r"(s - %d)" % j

        # termo bruto: Δ^k y0 / k! * produto
        Tk = Δk * prod_s / factorial(k)

        # separa numerador e denominador já simplificados
        num_Tk, den_Tk = fraction(Tk)
        num_Tk_exp = expand(num_Tk)

        # --- Saída em LaTeX para o termo k ---
        display(Math(r"\textbf{Termo }k=%d" % k))

        if k == 0:
            # k = 0: só y0
            display(Math(
                r"T_0(%s) = y_0 = %s" % (var_name, latex(Δk))
            ))
        else:
            # mostra Δ^k y0
            display(Math(
                r"\Delta^{%d} y_0 = %s" % (k, latex(Δk))
            ))

            # mostra produto em s
            display(Math(
                r"\prod_{j=0}^{%d-1}(s-j) = %s = %s"
                % (k, prod_s_tex, latex(expand(prod_s)))
            ))

            # mostra termo com k!
            display(Math(
                r"T_{%d}(%s) = \frac{\Delta^{%d} y_0}{%d!}\,%s"
                % (k, var_name, k, k, prod_s_tex or "1")
            ))

            # forma simplificada como fração única
            display(Math(
                r"T_{%d}(%s) = \frac{%s}{%s}"
                % (k, var_name, latex(num_Tk_exp), latex(den_Tk))
            ))

            # termo expandido em x
            display(Math(
                r"T_{%d}(%s) = %s"
                % (k, var_name, latex(expand(Tk)))
            ))

        P += Tk

    # 7) Polinômio final
    P_exp = expand(P)
    num_P, den_P = fraction(P_exp)
    num_P_exp = expand(num_P)

    display(Math(r"\textbf{Polinômio final}"))
    # forma expandida
    display(Math(r"P(%s) = %s" % (var_name, latex(P_exp))))
    # forma única fração, se tiver denominador
    if den_P != 1:
        display(Math(
            r"P(%s) = \dfrac{%s}{%s}"
            % (var_name, latex(num_P_exp), latex(den_P))
        ))

    return P_exp

if __name__ == "__main__":
    xs = [-9, -4, 1, 6, 11]     
    ys = [-808, -78, 2, 182, 1212]     

    P = gregory_newton_progressivo_latex(xs, ys)
    print("P(x) =", P)


Pontos de interpolação:
  x_0 = -9,  f(x_0) = -808
  x_1 = -4,  f(x_1) = -78
  x_2 = 1,  f(x_2) = 2
  x_3 = 6,  f(x_3) = 182
  x_4 = 11,  f(x_4) = 1212

Passo h = 5

Tabela de diferenças progressivas (somente primeira coluna):
  Δ^0 y_0 = -808
  Δ^1 y_0 = 730
  Δ^2 y_0 = -650
  Δ^3 y_0 = 750
  Δ^4 y_0 = 0



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

P(x) = x**3 - x**2 + 2
